Import Library

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.feature import graycomatrix, graycoprops
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Path Dataset

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/TUGAS ADIT/Smt 4/Comvis/dataset_daun"
SAVE_PATH    = "/content/drive/MyDrive/TUGAS ADIT/Smt 4/Comvis/dataset_daun/fitur_glcm_daun_bersih.csv"

Load Data

In [ ]:
def load_images(dataset_path):
    images, labels = [], []
    for label in sorted(os.listdir(dataset_path)):
        folder = os.path.join(dataset_path, label)
        if not os.path.isdir(folder):
            continue
        for file in sorted(os.listdir(folder)):
            if not file.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue
            img = cv2.imread(os.path.join(folder, file))
            if img is None:
                continue
            images.append(img)
            labels.append(label)
    print(f"Total gambar   : {len(images)}")
    print(f"Kelas          : {sorted(set(labels))}")
    return images, labels

pre processing

In [ ]:
def preprocess_images(images, labels):
    processed, new_labels = [], []
    for img, label in zip(images, labels):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(gray, (128, 128))
        processed.append(gray)
        new_labels.append(label)
    print(f"Total diproses : {len(processed)}")
    return processed, new_labels

ekstraksi fitur GLCM

In [ ]:
def extract_glcm_features(image):
    glcm = graycomatrix(image, distances=[1], angles=[0, np.pi/2],
                        levels=256, symmetric=True, normed=True)
    return [
        round(graycoprops(glcm, 'contrast').mean(),    4),
        round(graycoprops(glcm, 'correlation').mean(), 4),
        round(graycoprops(glcm, 'energy').mean(),      4),
        round(graycoprops(glcm, 'homogeneity').mean(), 4),
    ]

def extract_features(images):
    return [extract_glcm_features(img) for img in images]

Data Frame

In [ ]:
def create_dataframe(features, labels):
    df = pd.DataFrame(features, columns=['contrast', 'correlation', 'energy', 'homogeneity'])
    df['label'] = labels
    df = df.dropna().reset_index(drop=True)
    return df

Pipeline + tampil tabel + simpan CSV

In [ ]:
images, labels    = load_images(DATASET_PATH)
processed, labels = preprocess_images(images, labels)
features          = extract_features(processed)
df                = create_dataframe(features, labels)

print(f"\nShape : {df.shape}")
print(f"Distribusi kelas:\n{df['label'].value_counts()}\n")
display(df.head(220))

df.to_csv(SAVE_PATH, index=False)
print(f"\n✅ CSV tersimpan di:\n   {SAVE_PATH}")

Total gambar   : 220
Kelas          : ['Bawor', 'DuriHitam', 'MusangKing', 'SuperTembaga']
Total diproses : 220

Shape : (220, 5)
Distribusi kelas:
label
SuperTembaga    65
MusangKing      55
DuriHitam       50
Bawor           50
Name: count, dtype: int64



,contrast,correlation,energy,homogeneity,label
0,113.1363,0.9770,0.0610,0.4474,Bawor
1,117.3812,0.9693,0.0825,0.5004,Bawor
2,79.3477,0.9781,0.0901,0.5311,Bawor
3,157.0398,0.9755,0.0804,0.4792,Bawor
4,157.3992,0.9749,0.0823,0.4679,Bawor
...,...,...,...,...,...
215,229.8395,0.9720,0.0676,0.4310,SuperTembaga
216,176.6083,0.9708,0.1007,0.5494,SuperTembaga
217,150.1789,0.9746,0.0867,0.4914,SuperTembaga
218,122.9277,0.9703,0.0862,0.5053,SuperTembaga



✅ CSV tersimpan di:
   /content/drive/MyDrive/TUGAS ADIT/Smt 4/Comvis/dataset_daun/fitur_glcm_daun_bersih.csv


Split Data

In [ ]:
X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Normalisasi

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Training KNN

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=3)

evaluasi

In [ ]:
y_pred = knn.predict(X_test)

print("Akurasi:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Akurasi: 0.3409090909090909

Confusion Matrix:
 [[3 0 1 3]
 [4 5 0 2]
 [5 1 4 4]
 [4 2 3 3]]

Classification Report:
               precision    recall  f1-score   support

       Bawor       0.19      0.43      0.26         7
   DuriHitam       0.62      0.45      0.53        11
  MusangKing       0.50      0.29      0.36        14
SuperTembaga       0.25      0.25      0.25        12

    accuracy                           0.34        44
   macro avg       0.39      0.35      0.35        44
weighted avg       0.41      0.34      0.36        44

